In [1]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

THU_MUC_GOC = Path("..").resolve()
THU_MUC_DAC_TRUNG = THU_MUC_GOC / "data" / "features"
THU_MUC_MODEL = THU_MUC_GOC / "model"
THU_MUC_REPORT = THU_MUC_GOC / "report"

THU_MUC_MODEL.mkdir(parents=True, exist_ok=True)
THU_MUC_REPORT.mkdir(parents=True, exist_ok=True)

X_train = np.load(THU_MUC_DAC_TRUNG / "X_train.npy")
y_train = np.load(THU_MUC_DAC_TRUNG / "y_train.npy")
print(f"[LF] Đã nạp X_train: {X_train.shape} | y_train: {y_train.shape}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

bat_dau = time.time()
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=42))
])

param_lr = [
    {"lr__C": [0.0001, 0.0003, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.1, 1, 10]}
]
grid_lr = GridSearchCV(
    pipe_lr,
    param_grid=param_lr,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)
grid_lr.fit(X_train, y_train)
print(f"\nTìm xong tham số tốt nhất ({grid_lr.best_params_})")

thoi_gian = time.time() - bat_dau

lr_final = grid_lr.best_estimator_
joblib.dump(lr_final, THU_MUC_MODEL / "lr_model.pkl")

ket_qua = {
    "model": "LogisticRegression",
    "best_params": grid_lr.best_params_,
    "cv_f1_macro": round(float(grid_lr.best_score_), 4),
    "train_time_sec": round(thoi_gian, 1),
}

with open(THU_MUC_REPORT / "cv_lr.json", "w", encoding="utf-8") as f:
    json.dump(ket_qua, f, ensure_ascii=False, indent=2)

print("\nHoàn tất Logistic Regression:", ket_qua)

[LF] Đã nạp X_train: (12888, 1280) | y_train: (12888,)
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Tìm xong tham số tốt nhất ({'lr__C': 0.002})

Hoàn tất Logistic Regression: {'model': 'LogisticRegression', 'best_params': {'lr__C': 0.002}, 'cv_f1_macro': 0.9513, 'train_time_sec': 78.1}
